In [1]:

import pandas as pd
import numpy as np
import time
import math
from typing import Optional

In [14]:
class SimpleBatteryStorage:
    def __init__(self, states:np.ndarray, actions:np.ndarray, prices:np.ndarray, start_state:int, end_state:int):
        self.states = states.reshape(1, -1)
        self.actions = actions.reshape(1, -1)
        self.prices = prices.reshape(-1, 1)
        self.start_state = start_state
        self.end_state = end_state
        self.number_states = self.states.shape[-1]
        
    def get_state_transitions(self):
        state_base = (np.ones(shape=(self.states.shape[-1], self.actions.shape[-1])) * self.states.reshape(-1,1)).reshape(-1, 1)
        actions = np.tile(self.actions.T,(state_base.shape[0]//self.actions.T.shape[0],1))
        next_state = state_base + actions
        timestep_state_transition = np.concat([state_base, actions, next_state], axis=1)

        actions_clmn = timestep_state_transition[:,1][..., np.newaxis]
        prices = self.prices.T
        costs = (prices * actions_clmn).T.reshape(-1,1) 
        
        tiling = [1]*timestep_state_transition.shape[-1]
        tiling[0] = costs.shape[0]//timestep_state_transition.shape[0]
        state_transition = np.concat([np.tile(timestep_state_transition, reps=tiling).reshape(-1, 3), costs], axis=1)
        discharge_violation_ids = np.argwhere(state_transition[:, 2] < np.min(self.states)).squeeze()
        charge_violation_ids = np.argwhere(state_transition[:, 2] > np.max(self.states)).squeeze()
        
        state_transition[discharge_violation_ids, 2] = np.min(self.states)
        state_transition[charge_violation_ids, 2] = np.max(self.states)
        
        state_transition[np.concat([charge_violation_ids, discharge_violation_ids]), -1] = np.inf
        
        time_ids = np.tile(np.array(range(len(self.prices))), reps=[state_transition.shape[0]//len(self.prices),1])
        
        state_transition_matrix = np.concatenate([time_ids.T.reshape(-1, 1), state_transition], axis=1)
        return state_transition_matrix
    




class SimpleBatteryStorageChargesRestricted:
    def __init__(self, states:np.ndarray, actions:np.ndarray, prices:np.ndarray, start_state:int, end_state:int, max_charges:Optional[int]):
        self.states = states.reshape(-1, 1)
        self.actions = actions.reshape(1, -1)
        self.prices = prices
        self.start_state = start_state
        self.end_state = end_state
        self.max_charges = max_charges
        self.number_states = self.states.shape[0] * (self.max_charges+1) if max_charges is not None else self.states.shape[0]
        self.state_mapping = None
        
    def get_state_transitions(self):
        if self.max_charges is not None:
            skeleton_array = np.ones(shape=(self.states.shape[-1]*(self.max_charges+1)*self.actions.shape[-1]))
        else:
            skeleton_array = np.ones(shape=(self.states.shape[-1]*self.actions.shape[-1]))
            
        state_base = (skeleton_array*self.states).reshape(-1,1)

        charges_base = np.tile(np.arange(0, self.max_charges+1).reshape(-1,1), (self.states.shape[0], self.actions.shape[-1])).reshape(-1,1)
        _actions = np.resize(self.actions, (charges_base.shape[0], 1))

        next_state = np.concat([state_base+_actions, charges_base+np.abs(_actions)], axis=1)
        timestep_state_transition = np.concat([state_base, charges_base, _actions, next_state], axis=1)

        actions_clmn_id = 2

        actions_clmn = timestep_state_transition[:,actions_clmn_id][..., np.newaxis]
        prices = self.prices.T
        costs = (prices * actions_clmn).T.reshape(-1,1)

        state_transition = np.resize(timestep_state_transition, (costs.shape[0], timestep_state_transition.shape[1]))

        discharge_violation_ids = np.argwhere(state_transition[:, actions_clmn_id+1] < np.min(self.states)).squeeze()
        charge_violation_ids = np.argwhere(state_transition[:, actions_clmn_id+1] > np.max(self.states)).squeeze()

        charges_violation_ids = np.argwhere(state_transition[:, actions_clmn_id+2] > self.max_charges).squeeze()

        state_transition[discharge_violation_ids, actions_clmn_id+1] = np.min(self.states)
        state_transition[charge_violation_ids, actions_clmn_id+1] = np.max(self.states)
        state_transition[charges_violation_ids, actions_clmn_id+2] = self.max_charges

        _state_transition = np.concat([state_transition, costs], axis=1)
        state_transition = np.delete(_state_transition, np.unique(np.concat([discharge_violation_ids, charge_violation_ids, charges_violation_ids])), axis=0)
        #state_transition[np.unique(np.concat([discharge_violation_ids, charge_violation_ids, charges_violation_ids])), -1] = np.inf
        #time_ids = np.tile(np.array(range(len(prices))), reps=[state_transition.shape[0]//len(prices),1])
        
        #state_transition_matrix = np.concatenate([time_ids.T.reshape(-1, 1), state_transition], axis=1)
        #return state_transition_matrix
        return state_transition
    

    
    def map_states(self, state_transition_matrix):
        # Extract current and next state pairs
        current_states = state_transition_matrix[:, [0, 1]]
        next_states = state_transition_matrix[:, [3, 4]]
        
        # Combine both for unified unique indexing
        all_states = np.vstack((current_states, next_states))
        
        # Get unique states and their indices
        start = time.time()
        
        #unique_states, inverse_indices = np.unique(all_states, axis=0, return_inverse=True)
        # np.unique is slower than using pandas here
        df_states = pd.DataFrame(all_states).astype(int)
        unique_states_df = df_states.drop_duplicates().reset_index(drop=True).reset_index()
        inverse_indices = df_states.merge(unique_states_df, how='left', on=[0,1], sort=False)['index'].to_numpy()
        end = time.time()
        print(f'UNIQUE: {end-start}')
        
        start = time.time()
        #self.state_mapping = {tuple(state): idx for idx, state in enumerate(unique_states)} 
        #self.state_mapping = {tuple(state): idx for idx, state in enumerate(unique_states_df.loc[:, [0, 1]].to_numpy())}
        #self.state_mapping = {(state.0, state.1): state.index for _, state in unique_states_df.iterrows()}
        self.state_mapping = pd.Series(unique_states_df['index'].values, index=list(zip(unique_states_df[0], unique_states_df[1]))).to_dict()
        end = time.time()
        print(f'MAPPING DICT: {end-start}')
        
        #self.state_mapping = run_mapping(unique_states=unique_states)
        # Split back the indices
        current_state_ids = inverse_indices[:len(current_states)]
        next_state_ids = inverse_indices[len(current_states):]
        
        # Construct the final mapped matrix
        return np.column_stack((
            current_state_ids,
            state_transition_matrix[:, 2],
            next_state_ids, 
            state_transition_matrix[:, 5:]
        ))
        


In [46]:
prices = np.array([1,5])
prices = np.random.uniform(low=1, high=10, size=8760)
states = np.array([0, 1])
actions = np.array([-1, 0, 1])
bs = SimpleBatteryStorageChargesRestricted(states=states, actions=actions, prices=prices, start_state=0, end_state=0, max_charges=200)
bs_matrix = bs.map_states(bs.get_state_transitions())
#bs_matrix = bs.get_state_transitions()
bs_matrix = bs_matrix.reshape((len(prices), bs_matrix.shape[0]//len(prices), bs_matrix.shape[-1]))
bs_matrix[-1, ...]


UNIQUE: 3.1410186290740967
MAPPING DICT: 0.002870798110961914


array([[  0.        ,   0.        ,   0.        ,   0.        ],
       [  0.        ,   1.        , 202.        ,   4.15735903],
       [  1.        ,   0.        ,   1.        ,   0.        ],
       ...,
       [400.        ,  -1.        , 200.        ,  -4.15735903],
       [400.        ,   0.        , 400.        ,   0.        ],
       [401.        ,   0.        , 401.        ,   0.        ]],
      shape=(802, 4))

In [47]:

bs_matrix.shape


(8760, 802, 4)

In [64]:
# backward
end_state = None
value_list = []
for i,t in enumerate(reversed(range(bs_matrix.shape[0]))):
    if i == 0:
        if end_state is not None:
            previous_state_values = np.full((1, bs.number_states), np.inf)
            previous_state_values[0, end_state] = 0
        else:
            previous_state_values = np.zeros(shape=(1, bs.number_states))
    else:
        previous_state_values = value_list[i-1]
    
    # t_id = np.argwhere(bs_matrix[:, 0]==t).squeeze()
    # temp_array = bs_matrix[t_id,...]
    
    temp_array = bs_matrix[t, ...]
    
    temp_array = temp_array[np.argsort(temp_array[:, 0]), ...]
    
    temp_state_values = previous_state_values[:, np.int32(temp_array[:, -2])].squeeze()

    temp_state_values = temp_state_values + temp_array[:, -1]
    
    unique_states, idx_start = np.unique(temp_array[:, 0].astype(int), return_index=True)
    value_list.append(np.minimum.reduceat(temp_state_values, idx_start).reshape((1, bs.number_states)))
    #value_list.append(np.min(temp_state_values.reshape(bs.number_states, -1), axis=1).reshape((1, bs.number_states)))

In [65]:
value_list

[array([[ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.

In [66]:
value_list[0]

array([[ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0. 

In [67]:
# forward 

# aktuell wird der end_state nicht berücksichtigt. 
# Damit dieser berücksichtigt werden kann, muss direkt bei der Erzeugung der Matrix 
forward_value_list = list(reversed(value_list))
chosen_states = []
chosen_actions = []
for t in range(bs_matrix.shape[0]):
    # t_id = np.argwhere(bs_matrix[:, 0]==t).squeeze()
    temp_array = bs_matrix[t,...]
    
    if t == 0:
        if (bs.start_state is not None):
            temp_state = bs.start_state
        
        else:
            temp_state = np.argmin(forward_value_list[t].squeeze())
        chosen_states.append(temp_state)
    else:
        temp_state = chosen_states[t] # not t-1 since we already have the initial state in the chosen states list

    temp_array = temp_array[np.argwhere(temp_array[:, 0]==temp_state).squeeze(), ...].reshape((-1, temp_array.shape[-1]))
    
    if t == len(prices)-1:
        next_state_value = np.zeros(shape=(1, bs.number_states))
    else:
        next_state_value = forward_value_list[t+1]
    
    costs = temp_array[:, -1]
    diff_arr = (next_state_value[:, np.int32(temp_array[:, -2])] + costs).squeeze()
    
    action_selection = np.argmin(diff_arr).squeeze()
    
    chosen_actions.append(int(temp_array[action_selection, 1]))
    chosen_states.append(int(temp_array[action_selection, 2]))

In [68]:
chosen_states[-1]

200

In [69]:
chosen_actions

[0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 -1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 -1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0

In [22]:
bs.state_mapping


{(0, 0): 0,
 (0, 1): 1,
 (0, 2): 2,
 (0, 3): 3,
 (0, 4): 4,
 (0, 5): 5,
 (0, 6): 6,
 (0, 7): 7,
 (0, 8): 8,
 (0, 9): 9,
 (0, 10): 10,
 (0, 11): 11,
 (0, 12): 12,
 (0, 13): 13,
 (0, 14): 14,
 (0, 15): 15,
 (0, 16): 16,
 (0, 17): 17,
 (0, 18): 18,
 (0, 19): 19,
 (0, 20): 20,
 (0, 21): 21,
 (0, 22): 22,
 (0, 23): 23,
 (0, 24): 24,
 (0, 25): 25,
 (0, 26): 26,
 (0, 27): 27,
 (0, 28): 28,
 (0, 29): 29,
 (0, 30): 30,
 (0, 31): 31,
 (0, 32): 32,
 (0, 33): 33,
 (0, 34): 34,
 (0, 35): 35,
 (0, 36): 36,
 (0, 37): 37,
 (0, 38): 38,
 (0, 39): 39,
 (0, 40): 40,
 (0, 41): 41,
 (0, 42): 42,
 (0, 43): 43,
 (0, 44): 44,
 (0, 45): 45,
 (0, 46): 46,
 (0, 47): 47,
 (0, 48): 48,
 (0, 49): 49,
 (0, 50): 50,
 (0, 51): 51,
 (0, 52): 52,
 (0, 53): 53,
 (0, 54): 54,
 (0, 55): 55,
 (0, 56): 56,
 (0, 57): 57,
 (0, 58): 58,
 (0, 59): 59,
 (0, 60): 60,
 (0, 61): 61,
 (0, 62): 62,
 (0, 63): 63,
 (0, 64): 64,
 (0, 65): 65,
 (0, 66): 66,
 (0, 67): 67,
 (0, 68): 68,
 (0, 69): 69,
 (0, 70): 70,
 (0, 71): 71,
 (0, 72): 72

In [25]:
pd.DataFrame.from_dict(bs.state_mapping.keys(), index=bs.state_mapping.values())

TypeError: DataFrame.from_dict() got an unexpected keyword argument 'index'

In [31]:
reverse_df = pd.concat([pd.Series(bs.state_mapping.keys(), name='States'), pd.Series(bs.state_mapping.values(), name='Mapped States')], axis=1)
reverse_df


,States,Mapped States
0,"(0, 0)",0
1,"(0, 1)",1
2,"(0, 2)",2
3,"(0, 3)",3
4,"(0, 4)",4
...,...,...
397,"(1, 196)",397
398,"(1, 197)",398
399,"(1, 198)",399
400,"(1, 199)",400


In [37]:
reverse_df.iloc[chosen_states, :]['States'].to_list()

[(0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 (1, 1),
 

In [62]:
np.sum(np.array(chosen_actions) * prices)

np.float64(0.0)

In [74]:
prices = np.array([1,5,3,5])
#prices = np.random.uniform(low=1, high=10, size=8760)
states = np.array([0, 1])
actions = np.array([-1, 0, 1])
bs = SimpleBatteryStorage(states=states, actions=actions, prices=prices, start_state=0, end_state=0)
bs_matrix = bs.get_state_transitions()
bs_matrix

array([[ 0.,  0., -1.,  0., inf],
       [ 0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  1.,  1.,  1.],
       [ 0.,  1., -1.,  0., -1.],
       [ 0.,  1.,  0.,  1.,  0.],
       [ 0.,  1.,  1.,  1., inf],
       [ 1.,  0., -1.,  0., inf],
       [ 1.,  0.,  0.,  0.,  0.],
       [ 1.,  0.,  1.,  1.,  5.],
       [ 1.,  1., -1.,  0., -5.],
       [ 1.,  1.,  0.,  1.,  0.],
       [ 1.,  1.,  1.,  1., inf],
       [ 2.,  0., -1.,  0., inf],
       [ 2.,  0.,  0.,  0.,  0.],
       [ 2.,  0.,  1.,  1.,  3.],
       [ 2.,  1., -1.,  0., -3.],
       [ 2.,  1.,  0.,  1.,  0.],
       [ 2.,  1.,  1.,  1., inf],
       [ 3.,  0., -1.,  0., inf],
       [ 3.,  0.,  0.,  0.,  0.],
       [ 3.,  0.,  1.,  1.,  5.],
       [ 3.,  1., -1.,  0., -5.],
       [ 3.,  1.,  0.,  1.,  0.],
       [ 3.,  1.,  1.,  1., inf]])

In [80]:
# backward
value_list = []
number_state_action_pairs = bs_matrix.shape[0]//len(prices)
for i,t in enumerate(reversed(range(len(prices)))):
    if i == 0:
        previous_state_values = np.zeros(shape=(1, bs.states.shape[-1]))
    else:
        previous_state_values = value_list[i-1]
    
    t_id = np.argwhere(bs_matrix[:, 0]==t).squeeze()
    temp_array = bs_matrix[t_id,...]
    #print(temp_array)
    temp_state_values = previous_state_values[:, np.int32(temp_array[:, -2])].squeeze()

    temp_state_values = temp_state_values + temp_array[:, -1]
    
    value_list.append(np.min(temp_state_values.reshape(bs.states.shape[-1], -1), axis=1).reshape((1, bs.states.shape[-1])))
    
    

In [76]:
value_list

[array([[ 0., -5.]]),
 array([[-2., -5.]]),
 array([[-2., -7.]]),
 array([[-6., -7.]])]

In [77]:
# forward 
forward_value_list = list(reversed(value_list))
chosen_states = []
chosen_actions = []
for t in range(len(prices)):
    t_id = np.argwhere(bs_matrix[:, 0]==t).squeeze()
    temp_array = bs_matrix[t_id,...]
    
    if t == 0:
        if (bs.start_state is not None):
            temp_state = bs.start_state
        
        else:
            temp_state = np.argmin(forward_value_list[t].squeeze())
        chosen_states.append(temp_state)
    else:
        temp_state = chosen_states[t] # not t-1 since we already have the initial state in the chosen states list

    temp_array = temp_array[np.argwhere(temp_array[:, 1]==temp_state).squeeze(), ...]
    
    if t == len(prices)-1:
        next_state_value = np.zeros(shape=(1, bs.states.shape[-1]))
    else:
        next_state_value = forward_value_list[t+1]
    
    costs = temp_array[:, -1]
    diff_arr = (next_state_value[:, np.int32(temp_array[:, -2])] + costs).squeeze()
    
    action_selection = np.argmin(diff_arr).squeeze()
    
    chosen_actions.append(int(temp_array[action_selection, 2]))
    chosen_states.append(int(temp_array[action_selection, 3]))
    
    


In [78]:
chosen_states

[0, 1, 0, 1, 0]

In [79]:
chosen_actions

[1, -1, 1, -1]

In [9]:
np.sum(np.array(chosen_actions) * prices)

np.float64(-17729.731026335616)

In [3]:
from rivapy.optimization.models.dp_models.batterystorage import BatteryStorage
from rivapy.optimization.optimizer.dp import DPOptimizer
import numpy as np
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [147]:
states = np.array([0, 1])
actions = np.array([-1, 0, 1])
prices = np.array([1, 3, 2, 6, 0, 2])
max_charges = 2

bs = BatteryStorage(states=states, actions=actions, prices=prices, max_charges=2, start_level=0, start_charges=0, end_level=0, end_charges=2)

In [175]:
eff_in = 0.97
eff_out = 0.97
states = np.array([0, 1])
actions = [-1, 0, 1]
prices = np.array([1, 3, 2])
prices = np.random.uniform(low=1, high=10, size=8760)
max_charges = 200

# backward
value_matrix = np.zeros((len(prices), len(states), len(np.arange(0, max_charges+1))))
for i, t in enumerate(reversed(range(value_matrix.shape[0]))):
    if i == 0:
        continue # TODO: init für end states
    
    for state in range(len(states)):
        for charge in list(np.arange(0, max_charges+1)):
            value_list = []
            
            if charge == max_charges:
                value_matrix[t, state, charge] = value_matrix[t+1, state, charge]
                continue
            
            for action in actions:
                if action > 0:
                    next_state = state + eff_in*action
                    if next_state > np.max(states):
                        reward = np.inf
                        value = np.inf
                    else:
                        reward = action*prices[t]
                        # interpolation
                        ceil = int(math.ceil(next_state))
                        floor = int(math.floor(next_state))
                        
                        if floor != ceil:
                            m = (value_matrix[t+1, ceil, charge+1]-value_matrix[t+1, floor, charge+1])/(ceil-floor)
                            b = value_matrix[t+1, floor, charge+1] -m*floor
                                                        
                            value_next_state = m*next_state+b
                            value = value_next_state + reward
                        else:
                            value = value_matrix[t+1, ceil, charge+1] + reward
                        
                elif action < 0:
                    next_state = state+action
                    if next_state < np.min(states):
                        reward = np.inf
                        value = np.inf
                    else:
                        reward = action*eff_out*prices[t]
                        
                        ceil = int(math.ceil(next_state))
                        floor = int(math.floor(next_state))
                        if floor != ceil:
                            m = (value_matrix[t+1, ceil, charge+1]-value_matrix[t+1, floor, charge+1])/(ceil-floor)
                            b = value_matrix[t+1, floor, charge+1] -m*floor
                            
                            value_next_state = m*next_state+b 
                            value = value_next_state + reward
                        else:
                            value = value_matrix[t+1, ceil, charge+1] + reward
                else:
                    reward = 0
                    value = value_matrix[t+1, state, charge]

                value_list.append(value)
            
            #check if continous action could fill the storage
            if np.max(states) - state < np.max(actions)*eff_in:
                value = value_matrix[t+1, np.max(states), charge+1] + (np.max(states) - state)*eff_in*prices[t]
                value_list.append(value)
            
            value_matrix[t, state, charge] = np.min(value_list)
                

In [20]:
mapping = {
    0.0: 10,
    1.5: 20,
    3.0: 30,
    4.5: 40
}

# Convert to sorted arrays
keys = np.array(sorted(mapping.keys()))
values = np.array([mapping[k] for k in keys])

In [23]:
test = np.array([1,2,3,5])

In [36]:
np.searchsorted(test, 2)

np.int64(1)

In [21]:
@njit
def get_nearest_value(state, keys, values):
    idx = np.searchsorted(keys, state)

    if idx == 0:
        return values[0]
    elif idx == len(keys):
        return values[-1]
    else:
        # Check which neighbor is closer
        prev_key = keys[idx - 1]
        next_key = keys[idx]
        if abs(state - prev_key) <= abs(state - next_key):
            return values[idx - 1]
        else:
            return values[idx]

In [22]:
state = 2.2
mapped_value = get_nearest_value(state, keys, values)
print(mapped_value)

20


In [2]:
import numpy as np
from numba import njit
from numba.typed import Dict as numba_dict
from numba import types
# optimization suggestion: use the filtered state action pairs
@njit
def backward(eff_in, eff_out, state_map, states, actions, prices, max_charges, end_state=None):
    T = len(prices)   

    if end_state is None:
        value_matrix = np.zeros((T, len(states), max_charges+1))
    else:
        value_matrix = np.ones((T, len(states), max_charges+1)) * np.inf
        value_matrix[:, state_map[end_state], :] = 0
       
    for i in range(1, T):
        t = T - i - 1
        for state in states:
            for charge in range(max_charges+1):

                if charge == max_charges:
                    value_matrix[t, state_map[state], charge] = value_matrix[t+1, state_map[state], charge]
                    continue

                min_value = np.inf
                for action in actions:
                    if action > 0:
                        next_state = state + eff_in * action
                        if next_state > np.max(states):
                            reward = np.inf
                            value = np.inf
                        else:
                            reward = action * prices[t]
                            idx = np.searchsorted(states, next_state)
                            
                            if state_map.get(next_state, -1) != -1:
                                # ceil = state_map[next_state]
                                # floor = state_map[next_state]
                                value_next_state = value_matrix[t+1, state_map[next_state], charge+1]
                            else:
                                ceil = state_map[states[idx]]
                                floor = state_map[states[idx-1]]
                                
                                if value_matrix[t+1, ceil, charge+1] == np.inf or value_matrix[t+1, floor, charge+1] == np.inf:
                                    value_next_state = np.inf
                                    reward = np.inf
                                else:
                                    m = (value_matrix[t+1, ceil, charge+1] - value_matrix[t+1, floor, charge+1]) / (states[idx] - states[idx-1])
                                    b = value_matrix[t+1, floor, charge+1] - m * states[idx-1]
                                    value_next_state = m * next_state + b
                                    
                            # ceil = int(np.ceil(next_state))
                            # floor = int(np.floor(next_state))

                            # if floor != ceil:
                            #     if value_matrix[t+1, ceil, charge+1] == np.inf or value_matrix[t+1, floor, charge+1] == np.inf:
                            #         value_next_state = np.inf
                            #         reward = np.inf
                            #     else:
                            #         m = (value_matrix[t+1, ceil, charge+1] - value_matrix[t+1, floor, charge+1]) / (ceil - floor)
                            #         b = value_matrix[t+1, floor, charge+1] - m * floor
                            #         value_next_state = m * next_state + b
                            # else:
                            #     value_next_state = value_matrix[t+1, ceil, charge+1]

                            value = value_next_state + reward

                    elif action < 0:
                        next_state = state + action
                        if next_state < np.min(states):
                            reward = np.inf
                            value = np.inf
                        else:
                            reward = action * eff_out * prices[t]
                            
                            idx = np.searchsorted(states, next_state)
                            
                            if state_map.get(next_state, -1) != -1:
                                value_next_state = value_matrix[t+1, state_map[next_state], charge+1]
                            else:
                                ceil = state_map[states[idx]]
                                floor = state_map[states[idx-1]]
                                
                                if value_matrix[t+1, ceil, charge+1] == np.inf or value_matrix[t+1, floor, charge+1] == np.inf:
                                    value_next_state = np.inf
                                    reward = np.inf
                                else:
                                    m = (value_matrix[t+1, ceil, charge+1] - value_matrix[t+1, floor, charge+1]) / (states[idx] - states[idx-1])
                                    b = value_matrix[t+1, floor, charge+1] - m * states[idx-1]
                                    value_next_state = m * next_state + b
                            # if state_map.get(next_state, -1) != -1:
                            #     ceil = state_map[next_state]
                            #     floor = state_map[next_state]
                            # else:
                            #     ceil = state_map[states[idx]]
                            #     floor = state_map[states[idx-1]]
                            # # ceil = int(np.ceil(next_state))
                            # # floor = int(np.floor(next_state))

                            # if floor != ceil:
                            #     if value_matrix[t+1, ceil, charge+1] == np.inf or value_matrix[t+1, floor, charge+1] == np.inf:
                            #         value_next_state = np.inf
                            #         reward = np.inf
                            #     else:
                            #         m = (value_matrix[t+1, ceil, charge+1] - value_matrix[t+1, floor, charge+1]) / (ceil - floor)
                            #         b = value_matrix[t+1, floor, charge+1] - m * floor
                            #         value_next_state = m * next_state + b
                            # else:
                            #     value_next_state = value_matrix[t+1, ceil, charge+1]

                            value = value_next_state + reward
                    else:
                        reward = 0
                        value = value_matrix[t+1, state_map[state], charge]

                    if min_value > value:
                        min_value = value

                # Check if continuous action could fill the storage
                if (np.max(states) - state) <= (np.max(actions) * eff_in):
                    value = value_matrix[t+1, state_map[np.max(states)], charge+1] + (np.max(states) - state) * (1/eff_in) * prices[t]
                    
                    if min_value > value:
                        min_value = value
                
                # Check if continuous action could empty the storage
                if np.abs((np.min(states) - state)) <= np.abs(np.min(actions)):
                    value = value_matrix[t+1, state_map[np.min(states)], charge+1] - np.abs((np.min(states) - state)) * eff_out * prices[t]
                    
                    if min_value > value:
                        min_value = value
                        
                 # check if reaching the end state is possible and optimal
                if end_state is not None:
                    if end_state > state:
                        if (end_state - state) <= (np.max(actions) * eff_in):
                            value = value_matrix[t+1, state_map[end_state], charge+1] + (end_state - state) * (1/eff_in) * prices[t]
                    
                        if min_value > value:
                            min_value = value
                    
                    elif end_state < state:
                        if np.abs((end_state - state)) <= np.abs(np.min(actions)):
                            value = value_matrix[t+1, state_map[end_state], charge+1] - np.abs((end_state - state)) * eff_out * prices[t]
                            
                        if min_value > value:
                            min_value = value

                value_matrix[t, state_map[state], charge] = min_value

    return value_matrix

@njit
def forward(eff_in, eff_out, value_matrix, state_map, state_map_reverse, states, actions, prices, max_charges, start_state=None, start_charges=None, end_state=None):
    T = len(prices)
    
    state_choices = np.zeros(T)
    charges_choices = np.zeros(T)
    action_choices = np.zeros(T-1)
    
    prev_state = None
    prev_charge = None
    
    for t in range(T):
        value_matrix_slice = value_matrix[t]
        if t == 0:
            if start_state is None and start_charges is None:
                # min_index = np.argmin(value_matrix_slice)
                # prev_state, prev_charge = np.unravel_index(min_index, value_matrix_slice.shape)
                min_index = np.argmin(value_matrix_slice)
                rows, cols = value_matrix_slice.shape
                prev_state = min_index // cols  # row index
                prev_charge = min_index % cols 
                
                state_choices[t] = state_map_reverse[prev_state]
                charges_choices[t] = prev_charge
            
            elif start_charges is None:
                prev_state = start_state
                prev_charge = np.argmin(value_matrix_slice[state_map[start_state], :])
                
                state_choices[t] = prev_state
                charges_choices[t] = prev_charge
                
            elif start_state is None:
                prev_state = np.argmin(value_matrix_slice[:, start_charges])
                prev_charge = start_charges
                
                state_choices[t] = state_map_reverse[prev_state]
                charges_choices[t] = prev_charge
            else:
                prev_state = start_state
                prev_charge = start_charges
                
                state_choices[t] = prev_state
                charges_choices[t] = prev_charge
            
            continue
        
        prev_state = state_choices[t-1]
        prev_charge = int(charges_choices[t-1])
        
        if prev_charge == max_charges:
            action_choices[t-1] = 0
            state_choices[t] = prev_state
            charges_choices[t] = prev_charge
            continue
        
        best_value = np.inf
        
        chosen_state = None
        chosen_action = None
        
        for action in actions:
            if action > 0:
                next_state = prev_state + eff_in * action
                next_charge = prev_charge + 1
                if next_state > np.max(states):
                    reward = np.inf
                    value = np.inf
                else:
                    reward = action * prices[t-1]
                    idx = np.searchsorted(states, next_state)
                    
                    if state_map.get(next_state, None) is not None:
                        # ceil = state_map[next_state]
                        # floor = state_map[next_state]
                        value_next_state = value_matrix[t, state_map[next_state], prev_charge+1]
                    else:
                        ceil = state_map[states[idx]]
                        floor = state_map[states[idx-1]]
                        
                        if value_matrix[t, ceil, prev_charge+1] == np.inf or value_matrix[t, floor, prev_charge+1] == np.inf:
                            value_next_state = np.inf
                            reward = np.inf
                        else:
                            m = (value_matrix[t, ceil, prev_charge+1] - value_matrix[t, floor, prev_charge+1]) / (states[idx] - states[idx-1])
                            b = value_matrix[t, floor, prev_charge+1] - m * states[idx-1]
                            value_next_state = m * next_state + b
                    # ceil = int(np.ceil(next_state))
                    # floor = int(np.floor(next_state))

                    # if floor != ceil:
                    #     if value_matrix[t, ceil, prev_charge+1] == np.inf or value_matrix[t, floor, prev_charge+1] == np.inf:
                    #         value_next_state = np.inf
                    #         reward = np.inf
                    #     else:
                    #         m = (value_matrix[t, ceil, prev_charge+1] - value_matrix[t, floor, prev_charge+1]) / (ceil - floor)
                    #         b = value_matrix[t, floor, prev_charge+1] - m * floor
                    #         value_next_state = m * next_state + b
                    # else:
                    #     value_next_state = value_matrix[t, ceil, prev_charge+1]

                    value = value_next_state + reward

            elif action < 0:
                next_state = prev_state + action
                next_charge = prev_charge + 1
                if next_state < np.min(states):
                    reward = np.inf
                    value = np.inf
                else:
                    reward = action * eff_out * prices[t-1]
                    idx = np.searchsorted(states, next_state)
                    
                    if state_map.get(next_state, None) is not None:
                        # ceil = state_map[next_state]
                        # floor = state_map[next_state]
                        value_next_state = value_matrix[t, state_map[next_state], prev_charge+1]
                    else:
                        ceil = state_map[states[idx]]
                        floor = state_map[states[idx-1]]
                        
                        if value_matrix[t, ceil, prev_charge+1] == np.inf or value_matrix[t, floor, prev_charge+1] == np.inf:
                            value_next_state = np.inf
                            reward = np.inf
                        else:
                            m = (value_matrix[t, ceil, prev_charge+1] - value_matrix[t, floor, prev_charge+1]) / (states[idx] - states[idx-1])
                            b = value_matrix[t, floor, prev_charge+1] - m * states[idx-1]
                            value_next_state = m * next_state + b
                            
                    # if state_map.get(next_state, -1) != -1:
                    #     ceil = state_map[next_state]
                    #     floor = state_map[next_state]
                    # else:
                    #     ceil = state_map[states[idx]]
                    #     floor = state_map[states[idx-1]]
                    # # ceil = int(np.ceil(next_state))
                    # # floor = int(np.floor(next_state))

                    # if floor != ceil:
                    #     if value_matrix[t, ceil, prev_charge+1] == np.inf or value_matrix[t, floor, prev_charge+1] == np.inf:
                    #         value_next_state = np.inf
                    #         reward = np.inf
                    #     else:
                    #         m = (value_matrix[t, ceil, prev_charge+1] - value_matrix[t, floor, prev_charge+1]) / (ceil - floor) # CEIL AND FLOOR ARE WRONG HERE take states[idx] and states[idx-1] !!!!!!
                    #         b = value_matrix[t, floor, prev_charge+1] - m * floor
                    #         value_next_state = m * next_state + b
                    # else:
                    #     value_next_state = value_matrix[t, ceil, prev_charge+1]
                        

                    value = value_next_state + reward
            else:
                reward = 0
                next_state = prev_state
                next_charge = prev_charge
                idx = np.searchsorted(states, prev_state)
                if state_map.get(next_state, None) is not None:
                    # ceil = state_map[next_state]
                    # floor = state_map[next_state]
                    value_next_state = value_matrix[t, state_map[next_state], prev_charge+1]
                else:
                    ceil = state_map[states[idx]]
                    floor = state_map[states[idx-1]]
                    
                    if value_matrix[t, ceil, prev_charge+1] == np.inf or value_matrix[t, floor, prev_charge+1] == np.inf:
                        value_next_state = np.inf
                        reward = np.inf
                    else:
                        m = (value_matrix[t, ceil, prev_charge+1] - value_matrix[t, floor, prev_charge+1]) / (states[idx] - states[idx-1])
                        b = value_matrix[t, floor, prev_charge+1] - m * states[idx-1]
                        value_next_state = m * next_state + b
                # if state_map.get(prev_state, -1) != -1:
                #     ceil = state_map[prev_state]
                #     floor = state_map[prev_state]
                # else:
                #     ceil = state_map[states[idx]]
                #     floor = state_map[states[idx-1]]
                # # ceil = int(np.ceil(prev_state))
                # # floor = int(np.floor(prev_state))
                # if floor != ceil:
                #     if value_matrix[t, ceil, prev_charge] == np.inf or value_matrix[t, floor, prev_charge] == np.inf:
                #         value_next_state = np.inf
                #         reward = np.inf
                #     else:
                #         m = (value_matrix[t, ceil, prev_charge] - value_matrix[t, floor, prev_charge]) / (ceil - floor)
                #         b = value_matrix[t, floor, prev_charge] - m * floor
                #         value_next_state = m * next_state + b
                # else:
                #     value_next_state = value_matrix[t, ceil, prev_charge]

                value = value_next_state + reward
                #value = value_matrix[t, prev_state, prev_charge]


            if best_value > value:
                best_value = value
                chosen_state = next_state
                chosen_action = action
                chosen_charge = next_charge

        # Check if continuous action could fill the storage
        if (np.max(states) - prev_state) <= (np.max(actions) * eff_in):
            value = value_matrix[t, state_map[np.max(states)], prev_charge+1] + (np.max(states) - prev_state) * (1/eff_in) * prices[t-1]
            
            next_state = np.max(states)
            action = np.max(states) - prev_state
            next_charge = prev_charge + 1
            if best_value > value:
                best_value = value
                chosen_state = next_state
                chosen_action = action
                chosen_charge = next_charge
        
        # Check if continuous action could empty the storage
        if np.abs((np.min(states) - prev_state)) <= np.abs(np.min(actions)):
            value = value_matrix[t, state_map[np.min(states)], prev_charge+1] - np.abs((np.min(states) - prev_state)) * eff_out * prices[t-1]
            
            next_state = np.min(states)
            action = (-1)*np.abs((np.min(states) - prev_state))
            next_charge = prev_charge + 1
            if best_value > value:
                best_value = value
                chosen_state = next_state
                chosen_action = action
                chosen_charge = next_charge

        # check if reaching the end state is possible and optimal
        if end_state is not None:
            if end_state > prev_state:
                if (end_state - prev_state) <= (np.max(actions) * eff_in):
                    value = value_matrix[t, state_map[end_state], prev_charge+1] + (end_state - prev_state) * (1/eff_in) * prices[t-1]
            
                    next_state = end_state
                    action = end_state - prev_state
                    next_charge = prev_charge + 1
                    if best_value > value:
                        best_value = value
                        chosen_state = next_state
                        chosen_action = action
                        chosen_charge = next_charge
            
            elif end_state < prev_state:
                if np.abs((end_state - prev_state)) <= np.abs(np.min(actions)):
                    value = value_matrix[t, state_map[end_state], prev_charge+1] - np.abs((end_state - prev_state)) * eff_out * prices[t-1]
                    
                    next_state = end_state
                    action = end_state - prev_state
                    next_charge = prev_charge + 1
                    if best_value > value:
                        best_value = value
                        chosen_state = next_state
                        chosen_action = action
                        chosen_charge = next_charge
        
        state_choices[t] = chosen_state
        charges_choices[t] = chosen_charge
        action_choices[t-1] = chosen_action

            
    
    return state_choices, charges_choices, action_choices
        
        

In [105]:
@njit
def _value_wrapper(
    state: float,
    charge: int,
    action: float,
    max_state: float,
    min_state: float,
    prices: np.ndarray,
    t: int,
    states: np.ndarray,
    value_matrix: np.ndarray,
    eff_out: float,
    max_capacity: float,
    mode: int,
) -> float:
    if state < min_state or state > max_state:
        value = (-1) * 10 ** (12)
        return value
    if mode == 1:
        reward = (-1) * action / 100.0 * prices[t - 1] * max_capacity

    elif mode == -1:
        reward = (-1) * action / 100.0 * eff_out * prices[t - 1] * max_capacity

    elif mode == 0:
        reward = 0

    value = __value(state=state, charge=charge, t=t, states=states, value_matrix=value_matrix, reward=reward, mode=mode)
    return value

    # @staticmethod


@njit
def __value(
    state: float,
    charge: int,
    t: int,
    states: np.ndarray,
    value_matrix: np.ndarray,
    reward: float,
    mode: int,
) -> float:

    idx = np.searchsorted(states, state)
    mode_value = np.abs(mode)
    next_charge = charge * (1 - mode_value) + (charge + 1) * mode_value
    # if mode == 0:
    #     next_charge = charge
    # else:
    #     next_charge = charge + 1

    if np.abs(states[idx] - state) < 10 ** (-8):
        value_next_state = value_matrix[t, idx, next_charge]

    # elif np.abs(states[idx-1] - state) < 10**(-8):

    # if state_mapping.get(state, -1) != -1:
    #     value_next_state = value_matrix[t, state_mapping[state], next_charge]
    else:
        # ceil = state_mapping[states[idx]]
        # floor = state_mapping[states[idx - 1]]
        ceil = idx
        floor = idx - 1

        if value_matrix[t, ceil, next_charge] == (-1) * 10 ** (12) or value_matrix[t, floor, next_charge] == (-1) * 10 ** (12):
            # value_next_state = (-1) * 10**(12)
            # reward = (-1) * 10**(12)
            return (-1) * 10 ** (12)
        else:
            m = (value_matrix[t, ceil, next_charge] - value_matrix[t, floor, next_charge]) / (states[idx] - states[idx - 1])
            b = value_matrix[t, floor, next_charge] - m * states[idx - 1]
            value_next_state = m * state + b

    # print(value_next_state)
    return value_next_state + reward

    # @staticmethod


@njit
def backward(
    eff_in: float,
    eff_out: float,
    max_capacity: float,
    states: np.ndarray,
    actions: np.ndarray,
    prices: np.ndarray,
    max_charges: int,
    end_state: Optional[float] = None,
):
    T = len(prices)

    max_state = np.max(states)
    min_state = np.min(states)
    max_action = np.max(actions)
    min_action = np.min(actions)

    max_state_id = len(states) - 1

    if end_state is None:
        value_matrix = np.zeros((T, len(states), max_charges + 1))
    else:
        end_state_id = np.searchsorted(states, end_state)
        value_matrix = np.ones((T, len(states), max_charges + 1)) * 10 ** (12) * (-1)
        value_matrix[:, end_state_id, :] = 0

    for i in range(1, T):
        t = T - i - 1
        for state_id in range(len(states)):
            state = states[state_id]
            for charge in range(max_charges + 1):

                if charge == max_charges:
                    value_matrix[t, state_id, charge] = value_matrix[t + 1, state_id, charge]
                    continue

                min_value = 10 ** (12) * (-1)
                for action_id in range(len(actions)):
                    action = actions[action_id]
                    if action > 0:
                        next_state = state + eff_in * action
                        mode = 1

                    elif action < 0:
                        next_state = state + action
                        mode = -1

                    else:
                        next_state = state
                        mode = 0

                    value = _value_wrapper(
                        state=next_state,
                        charge=charge,
                        action=action,
                        max_state=max_state,
                        min_state=min_state,
                        prices=prices,
                        t=t + 1,
                        states=states,
                        value_matrix=value_matrix,
                        eff_out=eff_out,
                        max_capacity=max_capacity,
                        mode=mode,
                    )
                    if min_value < value:
                        min_value = value

                # Check if continuous action could fill the storage
                if (max_state - state) <= (max_action * eff_in):
                    value = value_matrix[t + 1, max_state_id, charge + 1] - (max_state - state) / 100.0 * (1 / eff_in) * prices[t] * max_capacity

                    if min_value < value:
                        min_value = value

                # Check if continuous action could empty the storage
                if np.abs((min_state - state)) <= np.abs(min_action):
                    value = value_matrix[t + 1, 0, charge + 1] + np.abs((min_state - state)) / 100.0 * eff_out * prices[t] * max_capacity

                    if min_value < value:
                        min_value = value

                # check if reaching the end state is possible and optimal
                if end_state is not None:
                    if end_state > state:
                        if (end_state - state) <= (max_action * eff_in):
                            value = (
                                value_matrix[t + 1, end_state_id, charge + 1] - (end_state - state) / 100.0 * (1 / eff_in) * prices[t] * max_capacity
                            )

                        if min_value < value:
                            min_value = value

                    elif end_state < state:
                        if np.abs((end_state - state)) <= np.abs(min_action):
                            value = (
                                value_matrix[t + 1, end_state_id, charge + 1]
                                + np.abs((end_state - state)) / 100.0 * eff_out * prices[t] * max_capacity
                            )

                        if min_value < value:
                            min_value = value

                value_matrix[t, state_id, charge] = min_value

    return value_matrix

    # @staticmethod


@njit
def forward(
    eff_in: float,
    eff_out: float,
    max_capacity: float,
    value_matrix: np.ndarray,
    states: np.ndarray,
    actions: np.ndarray,
    prices: np.ndarray,
    max_charges: int,
    start_state: Optional[float] = None,
    start_charges: Optional[float] = None,
    end_state: Optional[float] = None,
):
    T = len(prices)

    print_value = 0

    state_choices = np.zeros(T)
    charges_choices = np.zeros(T)
    objective = np.zeros(T - 1)
    action_choices = np.zeros(T - 1)

    max_state = np.max(states)
    min_state = np.min(states)
    max_action = np.max(actions)
    min_action = np.min(actions)

    max_state_id = len(states) - 1

    prev_state = None
    prev_charge = None

    if end_state is not None:
        end_state_id = np.searchsorted(states, end_state)

    for t in range(T):
        value_matrix_slice = value_matrix[t]
        if t == 0:
            if start_state is None and start_charges is None:
                # min_index = np.argmin(value_matrix_slice)
                # prev_state, prev_charge = np.unravel_index(min_index, value_matrix_slice.shape)
                min_index = np.argmax(value_matrix_slice)
                rows, cols = value_matrix_slice.shape
                prev_state = min_index // cols  # row index
                prev_charge = min_index % cols

                state_choices[t] = states[prev_state]
                charges_choices[t] = prev_charge

            elif start_charges is None:
                idx = np.searchsorted(states, start_state)
                prev_state = start_state
                prev_charge = np.argmax(value_matrix_slice[idx, :])

                state_choices[t] = prev_state
                charges_choices[t] = prev_charge

            elif start_state is None:
                prev_state_idx = np.argmax(value_matrix_slice[:, start_charges])
                prev_charge = start_charges

                state_choices[t] = states[prev_state_idx]
                charges_choices[t] = prev_charge
            else:
                prev_state = start_state
                prev_charge = start_charges

                state_choices[t] = prev_state
                charges_choices[t] = prev_charge

            continue

        prev_state = state_choices[t - 1]
        prev_charge = int(charges_choices[t - 1])

        if prev_charge == max_charges:
            action_choices[t - 1] = 0
            state_choices[t] = prev_state
            charges_choices[t] = prev_charge
            continue

        best_value = 10 ** (12) * (-1)

        chosen_state = None
        chosen_action = None

        for action_id in range(len(actions)):
            action = actions[action_id]
            if action > 0:
                next_state = prev_state + eff_in * action
                next_charge = prev_charge + 1
                mode = 1

            elif action < 0:
                next_state = prev_state + action
                next_charge = prev_charge + 1
                mode = -1

            else:
                next_state = prev_state
                next_charge = prev_charge
                mode = 0

            value = _value_wrapper(
                state=next_state,
                charge=prev_charge,
                action=action,
                max_state=max_state,
                min_state=min_state,
                prices=prices,
                t=t,
                states=states,
                value_matrix=value_matrix,
                eff_out=eff_out,
                max_capacity=max_capacity,
                mode=mode,
            )

            if best_value < value:
                best_value = value
                chosen_state = next_state
                chosen_action = action
                chosen_charge = next_charge

        # Check if continuous action could fill the storage
        if (max_state - prev_state) <= (max_action * eff_in):
            value = value_matrix[t, max_state_id, prev_charge + 1] - (max_state - prev_state) / 100.0 * (1 / eff_in) * prices[t - 1] * max_capacity

            next_state = max_state
            action = (max_state - prev_state) * (1 / eff_in)
            next_charge = prev_charge + 1
            if best_value < value:
                best_value = value
                chosen_state = next_state
                chosen_action = action
                chosen_charge = next_charge

        # Check if continuous action could empty the storage
        if np.abs((min_state - prev_state)) <= np.abs(min_action):
            value = value_matrix[t, 0, prev_charge + 1] + np.abs((min_state - prev_state)) / 100.0 * eff_out * prices[t - 1] * max_capacity

            next_state = min_state
            action = (-1) * np.abs((min_state - prev_state))
            next_charge = prev_charge + 1
            if best_value < value:
                best_value = value
                chosen_state = next_state
                chosen_action = action
                chosen_charge = next_charge

        # check if reaching the end state is possible and optimal
        if end_state is not None:
            if end_state > prev_state:
                if (end_state - prev_state) <= (max_action * eff_in):
                    value = (
                        value_matrix[t, end_state_id, prev_charge + 1]
                        - (end_state - prev_state) / 100.0 * (1 / eff_in) * prices[t - 1] * max_capacity
                    )

                    next_state = end_state
                    action = (end_state - prev_state) * (1 / eff_in)
                    next_charge = prev_charge + 1
                    if best_value < value:
                        best_value = value
                        chosen_state = next_state
                        chosen_action = action
                        chosen_charge = next_charge

            elif end_state < prev_state:
                if np.abs((end_state - prev_state)) <= np.abs(min_action):
                    value = (
                        value_matrix[t, end_state_id, prev_charge + 1]
                        + np.abs((end_state - prev_state)) / 100.0 * eff_out * prices[t - 1] * max_capacity
                    )

                    next_state = end_state
                    action = end_state - prev_state
                    next_charge = prev_charge + 1
                    if best_value < value:
                        best_value = value
                        chosen_state = next_state
                        chosen_action = action
                        chosen_charge = next_charge

        state_choices[t] = chosen_state
        charges_choices[t] = chosen_charge
        action_choices[t - 1] = chosen_action

        if chosen_action > 0:
            objective[t - 1] = chosen_action / 100.0 * prices[t - 1] * (-1) * max_capacity
        else:
            objective[t - 1] = chosen_action / 100.0 * eff_out * prices[t - 1] * (-1) * max_capacity

        # print(print_value)
    return state_choices, charges_choices, action_choices, objective

In [48]:
arr = np.array([1,2,5,6])

np.searchsorted(arr, 6)

np.int64(3)

# granular

In [109]:
eff_in = 0.97
eff_out = 0.97
np.random.seed(25)
#states = np.array([0, 1])
states = np.arange(0, 100.5, step=0.5)
# states = np.arange(0, 101)
#actions = np.array([-25, 0,25])
actions = np.arange(-25, 26)
prices = np.random.uniform(low=1, high=10, size=8)
max_charges = 200
max_capacity = 100

end_state = None
start_state = None
start_charge = None

# state_map = numba_dict.empty(
#     key_type=types.float64,
#     value_type=types.int64,
#     )

# state_map_reverse = numba_dict.empty(
#     key_type=types.int64,
#     value_type=types.float64,
#     )

# keys = states
# values = np.arange(len(states)+1)

# for i in range(len(keys)):
#     state_map[keys[i]] = values[i]
#     state_map_reverse[values[i]] = keys[i]

value_matrix = backward(eff_in, eff_out, max_capacity, states, actions, prices, max_charges, end_state=end_state)
state_choices, charges_choices, action_choices, objective = forward(eff_in, eff_out, max_capacity, value_matrix, states, actions, prices, max_charges, end_state=end_state, start_state=start_state, start_charges=start_charge)

In [108]:
np.sum(objective)

np.float64(684.2274634948717)

In [122]:
value_matrix.shape

(8, 201, 201)

In [62]:
np.max(value_matrix[0, ...])

np.float64(684.123216837796)

In [201]:
min_index = np.argmin(value_matrix[0, ...])
start_state, start_charge = np.unravel_index(min_index, value_matrix[0, ...].shape)
print(start_state, start_charge)

200 0


In [91]:
state_choices

array([ 99.,  99.,  99.,  99.,  99.,  99., 100., 100.])

In [92]:
action_choices

array([0.        , 0.        , 0.        , 0.        , 0.        ,
       1.03092784, 0.        ])

In [93]:
charges_choices

array([199., 199., 199., 199., 199., 199., 200., 200.])

In [199]:
0.97*(-25*8.83111723 - 25*6.24049236 - 25*3.50955047) + 1*2.67320109 - 0.97*25*4.69990115 + 25*2.05637992 -25*0.97*7.1647187

-684.2274637275

In [191]:
0.97*(-25*6.24049236 - 25*3.50955047) + 1*2.67320109 - 0.97*25*4.69990115 + 25*2.05637992 - 0.97*25*7.1647187 -24.72*0.97*4.93849954

-588.490188269936

In [190]:
prices

array([8.83111723, 6.24049236, 3.50955047, 2.67320109, 4.69990115,
       2.05637992, 7.1647187 , 4.93849954])

In [170]:
(-25*prices[0] - 25*prices[1] - 25*prices[2]-25*prices[4]  - 25*prices[6]) + 25*prices[5]

np.float64(-709.7349995122091)

In [165]:
0.97*(-25*prices[0] - 25*prices[1] - 25*prices[2]) + prices[3] -0.97*25*prices[4] + 25*prices[5] -25*0.97*prices[6]

np.float64(-684.2274634948718)

# coarse

In [16]:
eff_in = 0.97
eff_out = 0.97
np.random.seed(25)
states = np.array([0, 1])
states = np.arange(0, 101)
actions = np.array([-25, 0,25])
#actions = np.arange(-25, 26)
prices = np.random.uniform(low=1, high=10, size=8)
max_charges = 200

value_matrix = backward(eff_in, eff_out, states, actions, prices, max_charges)
state_choices, charges_choices, action_choices = forward(eff_in, eff_out, value_matrix, states, actions, prices, max_charges)

In [17]:
np.min(value_matrix[0, ...])

np.float64(-681.6883317298938)

In [154]:
state_choices

array([100.,  75.,  50.,  50.,  50.,  25.,  25.,   0.])

In [155]:
action_choices

array([-25., -25.,   0.,   0., -25.,   0., -25.])

In [59]:
prices

array([8.83111723, 6.24049236, 3.50955047, 2.67320109, 4.69990115,
       2.05637992, 7.1647187 , 4.93849954])

In [157]:
0.97*(-25*prices[0] - 25*prices[1] - 25*prices[4]-25*prices[6])# + prices[3] -0.97*25*prices[4] + 25*prices[5] -25*0.97*prices[6]

np.float64(-653.203563880986)

In [60]:
np.min(value_matrix[0, ...])

np.float64(-653.203563880986)

In [54]:
np.min(value_matrix[0, ...])

np.float64(-684.0906963713683)

In [220]:
value_matrix

array([[[-2.09433371e+04, -2.08263388e+04, -2.07330764e+04, ...,
         -3.55940254e+01,  0.00000000e+00,  0.00000000e+00],
        [-2.09445292e+04, -2.08470381e+04, -2.07342612e+04, ...,
         -2.17407608e+02,  0.00000000e+00,  0.00000000e+00],
        [-2.09460436e+04, -2.08677196e+04, -2.07357711e+04, ...,
         -2.17407608e+02,  0.00000000e+00,  0.00000000e+00],
        ...,
        [-2.14831160e+04, -2.13563621e+04, -2.12728593e+04, ...,
         -4.84871775e+02, -2.42453664e+02,  0.00000000e+00],
        [-2.14842854e+04, -2.13770226e+04, -2.12740207e+04, ...,
         -4.84871775e+02, -2.42453664e+02,  0.00000000e+00],
        [-2.14855014e+04, -2.13978909e+04, -2.12752296e+04, ...,
         -4.84871775e+02, -2.42453664e+02,  0.00000000e+00]],

       [[-2.09433371e+04, -2.08263388e+04, -2.07330764e+04, ...,
         -3.55940254e+01,  0.00000000e+00,  0.00000000e+00],
        [-2.09445292e+04, -2.08470381e+04, -2.07342612e+04, ...,
         -2.17407608e+02,  0.00000000e

In [156]:
bs.get_state_transition_matrix()

array([[[ 0.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  1.,  1.,  1.,  1.],
        [ 0.,  1.,  0.,  0.,  1.,  0.],
        [ 0.,  1.,  1.,  1.,  2.,  1.],
        [ 0.,  2.,  0.,  0.,  2.,  0.],
        [ 1.,  0., -1.,  0.,  1., -1.],
        [ 1.,  0.,  0.,  1.,  0.,  0.],
        [ 1.,  1., -1.,  0.,  2., -1.],
        [ 1.,  1.,  0.,  1.,  1.,  0.],
        [ 1.,  2.,  0.,  1.,  2.,  0.]],

       [[ 0.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  1.,  1.,  1.,  3.],
        [ 0.,  1.,  0.,  0.,  1.,  0.],
        [ 0.,  1.,  1.,  1.,  2.,  3.],
        [ 0.,  2.,  0.,  0.,  2.,  0.],
        [ 1.,  0., -1.,  0.,  1., -3.],
        [ 1.,  0.,  0.,  1.,  0.,  0.],
        [ 1.,  1., -1.,  0.,  2., -3.],
        [ 1.,  1.,  0.,  1.,  1.,  0.],
        [ 1.,  2.,  0.,  1.,  2.,  0.]],

       [[ 0.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  1.,  1.,  1.,  2.],
        [ 0.,  1.,  0.,  0.,  1.,  0.],
        [ 0.,  1.,  1.,  1.,  2.,  2.],
        [ 0.,  2.,  0.,  0.,  2.,  0

In [149]:
bs._internal_model.state_mapping_dict

{(0, 0): 0, (0, 1): 1, (0, 2): 2, (1, 0): 3, (1, 1): 4, (1, 2): 5}

In [150]:
dp = DPOptimizer(bs)

In [151]:
dp.backward()

In [152]:
dp._value_list

[array([[inf, inf,  0., inf, inf, inf]]),
 array([[inf, inf,  0., inf,  0., inf]]),
 array([[ 6., inf,  0., inf, -6., inf]]),
 array([[-4., inf,  0., inf, -6., inf]]),
 array([[-4., inf,  0., inf, -6., inf]]),
 array([[-5., inf,  0., inf, -6., inf]])]

In [153]:
dp.forward()

In [154]:
dp._chosen_states

[0, 4, 4, 4, 2, 2]

In [155]:
dp._chosen_actions

[1, 0, 0, -1, 0]